In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from umap import UMAP
import matplotlib.pyplot as plt
import pandas as pd
from scipy import linalg
import torch
from sklearn.manifold import TSNE
import umap.plot
from sklearn.preprocessing  import MinMaxScaler
from collections import Counter

/doctorai/marinafr/progs/miniconda3/envs/airr_atlas/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.
    Params:
    -- mu1:    The mean of the activations of preultimate layer of the
               CHEMNET (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2:    The mean of the activations of preultimate layer of the
               CHEMNET (like returned by the function 'get_predictions')
               for real samples.
    -- sigma1: The covariance matrix of the activations of preultimate layer
               of the CHEMNET (like returned by the function 'get_predictions')
               for generated samples.
    -- sigma2: The covariance matrix of the activations of preultimate layer
               of the CHEMNET (like returned by the function 'get_predictions')
               for real samples.
    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert (
        mu1.shape == mu2.shape
    ), "Training and test mean vectors have different lengths"
    assert (
        sigma1.shape == sigma2.shape
    ), "Training and test covariances have different dimensions"

    diff = mu1 - mu2

    # product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            #raise ValueError("Imaginary component {}".format(m))
            print("Imaginary component {}".format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * tr_covmean

In [5]:
def get_w2(act1, act2):

    """Calculate w2 between two sets

    Args:
        act1: First set
        act2: Second set

    Returns:
        float: The FCD score
    """

    mu1 = np.mean(act1, axis=0)
    sigma1 = np.cov(act1.T)

    mu2 = np.mean(act2, axis=0)
    sigma2 = np.cov(act2.T)

    fcd_score = calculate_frechet_distance(
        mu1=mu1, mu2=mu2, sigma1=sigma1, sigma2=sigma2
    )

    return fcd_score

In [6]:
def get_pairwise_w2(datasets):
    # Initialize a list to store distance matrices
    distance_matrices = []

    # Calculate distance matrices for all unique combinations of datasets
    n_datasets = len(datasets)
    for i in range(n_datasets):
        for j in range(i, n_datasets):
            dist_matrix = get_w2(datasets[i], datasets[j])
            distance_matrices.append(dist_matrix)

    return distance_matrices

In [7]:
from scipy.spatial.distance import squareform
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap, LogNorm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def build_plot(distance_matrices, model):
    # Define the colors for the colormap
    colors = ["#023047", "#219EBC", "#8ECAE6"] # Dark blue to light blue

    # Create the colormap
    cmap = LinearSegmentedColormap.from_list("my_colormap", colors)

    n_datasets = 5
    dataset_size = 790000

    # Create a square heatmap
    fig, ax = plt.subplots(figsize=(12, 12))

    # Create the combined distance matrix for the square heatmap
    combined_matrix = np.zeros((n_datasets, n_datasets))
    idx = 0
    for i in range(n_datasets):
        for j in range(i, n_datasets):
            combined_matrix[i, j] = distance_matrices[idx]
            combined_matrix[j, i] = combined_matrix[i, j]
            idx += 1

    #new_order = [0, 1, 3, 4, 2, 5]

    # Use np.ix_ to reorder rows and columns simultaneously
    #combined_matrix = combined_matrix[np.ix_(new_order, new_order)]

    # Create the heatmap
    sns.heatmap(combined_matrix, cmap=cmap, annot=True, annot_kws={"size": 16}, cbar=False, square=True)
    #sns.heatmap(combined_matrix, cmap="coolwarm", annot=False, square=True, cbar_kws={'label': 'W2 Distance'})  # Add cbar_kws

    # Customize the labels
    ax.set_xticklabels(['Wang H-CDR3', 'iReceptor', 'Random aa', 'Random shuffled', 'Random'], fontsize=18, rotation=45)
    ax.set_yticklabels(['Wang H-CDR3', 'iReceptor', 'Random aa', 'Random shuffled', 'Random'], fontsize=18, rotation=45)
    ax.set_title(f"W2 between datasets, {model.upper()}, N = {dataset_size}", fontsize=20)

    plt.tight_layout()
    plt.savefig(f'w2_heatmap_{model}_790000')

    # Show the heatmap
    plt.show()

In [9]:
esm2_df1 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()
esm2_df2 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()
esm2_df3 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()[:790000]
esm2_df4 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()[:790000]
esm2_df5 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()[:790000]

ab2_df1 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()
ab2_df2 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()
ab2_df3 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()[:790000]
ab2_df4 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()[:790000]
ab2_df5 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()[:790000]

ohe_df1 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()
ohe_df2 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()
ohe_df3 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()[:790000]
ohe_df4 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()[:790000]
ohe_df5 = torch.load(f'<UPDATE_PATH>'  # TODO: update path to your data).numpy()[:790000]


In [10]:
esm2_datasets = [esm2_df1, esm2_df2, esm2_df3, esm2_df4, esm2_df5]
ab2_datasets = [ab2_df1, ab2_df2, ab2_df3, ab2_df4, ab2_df5]
ohe_datasets = [ohe_df1, ohe_df2, ohe_df3, ohe_df4, ohe_df5]

In [ ]:
esm2_distance_matrices = get_pairwise_w2(esm2_datasets)
esm2_distance_matrices
#torch.save(torch.tensor(esm2_distance_matrices), f'./w2_esm2.pt')

[-8.536016338211994e-10,
 0.015371392771540648,
 0.11600123880141311,
 0.12271449954302938,
 1.2775181016964243,
 -5.574856132284367e-10,
 0.11868859093078044,
 0.11084402492159917,
 1.321246832152748,
 -1.7763568394002505e-14,
 0.12455563191860364,
 1.1447792279976543,
 -9.069669459904617e-10,
 1.2031630985567734,
 -5.673719272181188e-10]

In [18]:
torch.save(torch.tensor(esm2_distance_matrices), f'./w2_esm2.pt')

In [19]:
ab2_distance_matrices = get_pairwise_w2(ab2_datasets)
torch.save(torch.tensor(ab2_distance_matrices), f'./w2_ab2.pt')

In [11]:
for i, arr in enumerate(ohe_datasets, 1):
    print(i, arr.shape)

1 (866990, 660)
2 (799903, 660)
3 (790000, 660)
4 (790000, 660)
5 (790000, 660)


In [ ]:
ohe_distance_matrices = get_pairwise_w2(ohe_datasets)
torch.save(torch.tensor(ohe_distance_matrices), f'./w2_ohe.pt')

In [ ]:
#build_plot(esm2_distance_matrices, 'esm2')
#build_plot(ab2_distance_matrices, 'ab2')
build_plot(ohe_distance_matrices, 'ohe')

NameError: name 'esm2_distance_matrices' is not defined